## 1. Load Metadata, Splits, and DataLoaders

This cell loads the prebuilt `data_pipeline.pkl`, extracts all dataset metadata, and builds the train/validation/test datasets and loaders used by the rest of the notebook.


In [1]:
import sys, os
import pickle
import random
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights

from google.colab import drive
drive.mount('/content/drive')

_DB_ROOT   = '/content/drive/MyDrive/XAI-Project/DB'
DRIVE_ROOT = f'{_DB_ROOT}/DB1'
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints'
PKL_DIR    = f'{DRIVE_ROOT}/pipeline'
DATA_PKL   = f'{PKL_DIR}/data_pipeline_resnet.pkl'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(PKL_DIR, exist_ok=True)
print('Google Drive mounted.')

print(f'DATA_PKL  : {DATA_PKL}')

with open(DATA_PKL, 'rb') as f:
    bundle = pickle.load(f)

paths = bundle['paths']
splits = bundle['splits']
meta = bundle['meta']

DATASET_DIR       = paths['dataset_dir']
ATTR_TXT          = paths['attr_txt']
NUM_L1            = meta['NUM_L1']
NUM_L2            = meta['NUM_L2']
CONCEPT_NAMES     = meta['CONCEPT_NAMES']
attr_parent_idx   = meta['attr_parent_idx']
part_to_attr_indices = meta['part_to_attr_indices']
part_to_idx       = meta['part_to_idx']
attr_names        = meta['attr_names']
class_names_cub   = meta['class_names_cub']
# IMG_SIZE: 336 — sweet spot for CUB-200 with ResNet-50 V2.
IMG_SIZE          = 336
# BATCH_SIZE: 96 — fits H100 80GB at 336² with ResNet-50 + BF16.
BATCH_SIZE        = 96

train_images = splits['train']['images']
train_labels = splits['train']['labels']
train_l1      = np.asarray(splits['train']['l1'], dtype=np.float32)
train_l2      = np.asarray(splits['train']['l2'], dtype=np.float32)
train_certs   = np.asarray(splits['train']['cert'], dtype=np.float32)
train_vis     = np.asarray(splits['train']['vis'], dtype=np.float32)
train_visibility = train_vis
train_ids     = splits['train']['ids']

val_images = splits['val']['images']
val_labels = splits['val']['labels']
val_l1     = np.asarray(splits['val']['l1'], dtype=np.float32)
val_l2     = np.asarray(splits['val']['l2'], dtype=np.float32)
val_certs  = np.asarray(splits['val']['cert'], dtype=np.float32)
val_vis    = np.asarray(splits['val']['vis'], dtype=np.float32)
val_visibility = val_vis
val_ids    = splits['val']['ids']

test_images = splits['test']['images']
test_labels = splits['test']['labels']
test_l1     = np.asarray(splits['test']['l1'], dtype=np.float32)
test_l2     = np.asarray(splits['test']['l2'], dtype=np.float32)
test_certs  = np.asarray(splits['test']['cert'], dtype=np.float32)
test_vis    = np.asarray(splits['test']['vis'], dtype=np.float32)
test_visibility = test_vis
test_ids    = splits['test']['ids']

# ── Training augmentation ─────────────────────────────────────────────────────
# Stronger pipeline for CUB-200 at 336²:
#   - RandomResizedCrop scale [0.5, 1.0] (not as aggressive at 336 — bird is small)
#   - RandAugment: 2 ops, magnitude 9 — adds Sharpness, AutoContrast, ShearX/Y, etc.
#   - ColorJitter + Rotation + RandomErasing kept for cutout-style regularisation
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0), ratio=(0.75, 1.33)),
    transforms.RandomHorizontalFlip(),
    transforms.RandAugment(num_ops=2, magnitude=9),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.3, scale=(0.02, 0.2), ratio=(0.3, 3.3)),
])

# ── Evaluation transform: deterministic centre crop ───────────────────────────
# Resize 384 → CenterCrop 336 keeps the same ~87.5% crop ratio used at 224 (256→224).
eval_transform = transforms.Compose([
    transforms.Resize(384),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


class BirdDataset(Dataset):
    def __init__(self, image_paths, labels, l1_labels, l2_labels, certainty, visibility,
                 img_root, transform=None):
        self.paths = image_paths
        self.labels = labels
        self.l1 = l1_labels
        self.l2 = l2_labels
        self.cert = certainty
        self.vis = visibility
        self.img_root = img_root
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(
            os.path.join(self.img_root, 'images', self.paths[idx])
        ).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.labels[idx] - 1, dtype=torch.long)
        l1    = torch.tensor(self.l1[idx], dtype=torch.float32)
        l2    = torch.tensor(self.l2[idx], dtype=torch.float32)
        cert  = torch.tensor(self.cert[idx], dtype=torch.float32)
        vis   = torch.tensor(self.vis[idx], dtype=torch.float32)
        mask  = torch.zeros(1, IMG_SIZE, IMG_SIZE)

        return img, label, l1, l2, mask, cert, vis


# At 336² with batch 96 and stronger augmentation, CPU decoding is the bottleneck;
# 12 workers + prefetch_factor=4 keeps the H100 fed.
NUM_WORKERS = 12

train_dataset = BirdDataset(
    train_images, train_labels, train_l1, train_l2, train_certs, train_vis,
    DATASET_DIR, train_transform
)
val_dataset = BirdDataset(
    val_images, val_labels, val_l1, val_l2, val_certs, val_vis,
    DATASET_DIR, eval_transform
)
test_dataset = BirdDataset(
    test_images, test_labels, test_l1, test_l2, test_certs, test_vis,
    DATASET_DIR, eval_transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4, drop_last=False)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4, drop_last=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          persistent_workers=True, prefetch_factor=4, drop_last=False)

print(f'IMG_SIZE      : {IMG_SIZE}   BATCH_SIZE: {BATCH_SIZE}')
print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')
print(f'NUM_L1={NUM_L1}, NUM_L2={NUM_L2}')
print(f'CKPT_DIR : {CKPT_DIR}')
print(f'PKL_DIR  : {PKL_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted.
DATA_PKL  : /content/drive/MyDrive/XAI-Project/DB/DB1/pipeline/data_pipeline_resnet.pkl
IMG_SIZE      : 336   BATCH_SIZE: 96
Train batches : 57
Val batches   : 7
Test batches  : 61
NUM_L1=13, NUM_L2=312
CKPT_DIR : /content/drive/MyDrive/XAI-Project/DB/DB1/checkpoints
PKL_DIR  : /content/drive/MyDrive/XAI-Project/DB/DB1/pipeline


## 2. Define the H-CBM Architecture

This cell defines the Hierarchical Concept Bottleneck Model, including the ResNet-50 backbone, the coarse and fine concept heads, the hierarchical masking logic, and a quick forward-pass sanity check.


In [2]:
class HierarchicalCBM(nn.Module):
    """
    Hierarchical Concept Bottleneck Model (H-CBM).

    Architecture:
        Backbone    : ResNet-50 (pretrained ImageNet V2) → Feature Map F (2048,)
        Coarse Head : g_c(F)          → z_c (NUM_L1,) logits, p_c (NUM_L1,) probs
        Fine Head   : g_f(F, p_c)     → z_f (NUM_L2,) logits, p_f (NUM_L2,) probs
                      with Masked Fine Head: p_f[i] = σ(z_f[i]) × p_c[parent(i)]
        Classifier  : h(p_f)          → (num_classes,)  — reads ONLY from p_f

    Key properties:
        - Bottleneck: classifier never sees raw features — only concepts
        - Masked Fine Head: hard architectural hierarchy constraint
        - attr_parent_idx: maps each L2 attribute to its L1 parent (-1 = no parent)
        - BatchNorm1d in concept heads: stable training, faster convergence
        - detach_coarse: when True, stops gradient from fine_head flowing into coarse_head
          (used in Phase 1 to prevent the 312-attribute gradient from corrupting coarse)
    """

    def __init__(
        self,
        attr_parent_idx: np.ndarray,
        num_classes: int = 200,
        num_l1: int = 13,
        num_l2: int = 312,
    ):
        super().__init__()

        self.num_l1 = num_l1
        self.num_l2 = num_l2

        self.register_buffer(
            'attr_parent_idx',
            torch.tensor(attr_parent_idx, dtype=torch.long)
        )

        # ResNet-50 V2: stronger ImageNet pre-training (~82% top-1 vs ~76% V1).
        # Output dim 2048.
        backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        self.features = nn.Sequential(*list(backbone.children())[:-1])

        # Coarse head: 2048 → 256 (BN + ReLU + Dropout) → num_l1
        # 256 hidden units: prevents overfitting for 13 binary outputs from frozen features
        # Dropout=0.2 (not 0.4): lower stochasticity → val tracks train; WD=1e-3 handles regularisation
        self.coarse_head = nn.Sequential(
            nn.Linear(2048, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_l1),
        )

        # Fine head: (2048 + num_l1) → 512 (BN + ReLU + Dropout) → num_l2
        # 512 hidden units: sufficient capacity for 312 outputs without excessive overfit
        self.fine_head = nn.Sequential(
            nn.Linear(2048 + num_l1, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, num_l2),
        )

        self.classifier = nn.Linear(num_l2, num_classes)

    def forward(self, x: torch.Tensor, detach_coarse: bool = False):
        feats = self.features(x).flatten(1)
        z_c = self.coarse_head(feats)
        p_c = torch.sigmoid(z_c)

        # detach_coarse=True in Phase 1: stops L_fine gradient from
        # flowing back through p_c into coarse_head (prevents gradient interference)
        p_c_for_fine = p_c.detach() if detach_coarse else p_c
        fine_input = torch.cat([feats, p_c_for_fine], dim=1)
        z_f = self.fine_head(fine_input)
        p_f_raw = torch.sigmoid(z_f)

        parent_idx = self.attr_parent_idx
        has_parent = parent_idx >= 0
        safe_idx = parent_idx.clamp(min=0)
        parent_probs = p_c[:, safe_idx]
        # Soft hierarchical mask: mask ∈ [0.5, 1.0] instead of [0, 1].
        # Preserves the hierarchy signal (mask scales with parent confidence)
        # but prevents p_f → 0 early in training when p_c ≈ 0.5, which
        # otherwise starves the classifier of concept signal.
        soft_parent = 0.5 + 0.5 * parent_probs
        mask = torch.where(
            has_parent.unsqueeze(0),
            soft_parent,
            torch.ones_like(parent_probs),
        )
        p_f = p_f_raw * mask

        cls_logits = self.classifier(p_f)
        return cls_logits, z_c, p_c, z_f, p_f


model = HierarchicalCBM(
    attr_parent_idx=attr_parent_idx,
    num_classes=200,
    num_l1=NUM_L1,
    num_l2=NUM_L2,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print('HierarchicalCBM class defined.')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('Running on CPU — OK for model definition and forward pass check')
    print('GPU needed for training')

# Sanity check — also verify detach_coarse does not change output values
model.eval()
with torch.no_grad():
    dummy_input = torch.randn(2, 3, IMG_SIZE, IMG_SIZE, device=device)
    out_normal  = model(dummy_input, detach_coarse=False)
    out_detach  = model(dummy_input, detach_coarse=True)
    assert torch.allclose(out_normal[0], out_detach[0]), 'detach_coarse changed output!'
print(f'Forward pass OK: cls={tuple(out_normal[0].shape)}, z_c={tuple(out_normal[1].shape)}, z_f={tuple(out_normal[3].shape)}')
print('detach_coarse sanity check passed.')

model.train()

HierarchicalCBM class defined.
Device : cuda
GPU    : NVIDIA RTX PRO 6000 Blackwell Server Edition
Forward pass OK: cls=(2, 200), z_c=(2, 13), z_f=(2, 312)
detach_coarse sanity check passed.


HierarchicalCBM(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (0): Co

## 3. Training Configuration


In [3]:
import random, time
import torch.nn.functional as F

# ─── Training Checkpoint & History Paths ──────────────────────────────────────
CKPT_PATH    = os.path.join(CKPT_DIR, 'best_model.pth')
HISTORY_PATH = os.path.join(PKL_DIR, 'train_history.pkl')

# ─── Learning Rates & Regularisation ──────────────────────────────────────────
# BACKBONE_LR: 5e-5 — CUB has only ~60 images/class and bird textures differ
#   enough from ImageNet that the backbone needs real adaptation. 1e-5 was too
#   conservative (frozen-like behaviour); 5e-5 lets ResNet-50 specialise without
#   catastrophic forgetting.
BACKBONE_LR     = 5e-5

# HEADS_LR: 1e-4 — lowered from 3e-4 for stability when the backbone is jointly
#   trained at 5e-5. Ratio HEADS_LR/BACKBONE_LR = 2 keeps heads ahead of backbone
#   without oscillation.
HEADS_LR        = 1e-4

# WEIGHT_DECAY: 1e-3 — CUB has ~60 training images/class → strong L2 regularisation
#   is critical.  1e-3 is the most effective single anti-overfitting lever here.
WEIGHT_DECAY    = 1e-3

# ─── Phase-3-specific overrides ───────────────────────────────────────────────
# After P1+P2, the model is already at ~62% val_acc with backbone @ 5e-5. Joint
# refinement (P3) then needs MUCH lower LRs and stronger WD; otherwise the model
# overfits in the first ~10 epochs (the symptom we observed: best epoch=10).
PHASE3_BACKBONE_LR = 1e-5     # ↓ from 5e-5
PHASE3_HEADS_LR    = 3e-5     # ↓ from 1e-4
PHASE3_WD          = 5e-3     # ↑ from 1e-3 (all params unfrozen → more capacity to overfit)

# ─── Loss Weights (Phase 1 / Phase 3 base) ────────────────────────────────────
# Equal weighting in P3 (overridden by PHASE3_LAMBDA_* below); kept here as the
# canonical "default" referenced by sanity-check cells.
LAMBDA_COARSE   = 1.0
LAMBDA_FINE     = 1.0
LAMBDA_TASK     = 1.0

# Phase 3-specific weights: keep concept supervision DOMINANT (×2) so the
# concept space stays meaningful and well-calibrated; classifier was already
# trained in P2 so it doesn't need extra task signal. This is the key fix for
# the "P3 best at epoch 10 then overfit" pattern.
PHASE3_LAMBDA_COARSE = 2.0
PHASE3_LAMBDA_FINE   = 2.0
PHASE3_LAMBDA_TASK   = 1.0

# LAMBDA_TASK_P1=0.1 — small task signal during Phase 1 so the concept heads
#   learn representations that are ALSO useful for classification, instead of
#   purely optimising binary concept BCE/Focal. This shapes p_f to be more
#   discriminative when Phase 3 begins → faster convergence to higher val_acc.
#   The classifier is also trainable in P1 (still linear, still bottleneck).
LAMBDA_TASK_P1  = 0.1

# ─── Training Stability ───────────────────────────────────────────────────────
# LABEL_SMOOTHING=0.1: standard value; prevents overconfidence on 200 fine-grained classes.
LABEL_SMOOTHING = 0.1
GRAD_CLIP       = 1.0

# WARMUP_EPOCHS=5 (↓ from 10): the model already starts P3 in good shape after
#   P1+P2; long warmup wastes epochs and lets the model overfit before LR peaks.
WARMUP_EPOCHS   = 5

# ─── Phase 1 / 2 Scheduler (ReduceLROnPlateau) ────────────────────────────────
# LR_PATIENCE=7 (↑ from 5): avoids reducing LR too aggressively early in training
#   when loss is still noisy.  Factor=0.3 is kept — aggressive enough to escape plateau.
LR_PATIENCE    = 7
LR_FACTOR      = 0.3

# ─── Early Stopping ───────────────────────────────────────────────────────────
# PHASE1_PATIENCE=20: Phase 1 trains BOTH heads jointly (no detach), so it
#   converges on a combined L_coarse + L_fine plateau. 20 ep buffer accommodates
#   the slower-converging fine head (312 outputs, focal loss).
PHASE1_PATIENCE   = 20

# EARLY_STOP_PAT=50 (↑ from 30): with the much lower P3 LR (1e-5/3e-5), cosine
#   decay produces slow but steady improvements; need more patience to let the
#   schedule finish.
EARLY_STOP_PAT    = 50

# OVERFIT_THRESHOLD=0.20 (↑ from 0.12): at 336² with strong augmentation, train
#   acc naturally exceeds val acc by ~15% even without true overfitting; bumping
#   to 0.20 prevents premature stopping while still catching real overfit.
OVERFIT_THRESHOLD = 0.20

# OVERFIT_PATIENCE=15 (↑ from 8): only stop if the gap is truly persistent.
OVERFIT_PATIENCE  = 15

# ─── Phase Epoch Counts (upper bounds — EarlyStopper handles actual stopping) ──
SEED             = 42

# PHASE1_EPOCHS=120: joint head training (coarse + fine simultaneously, backbone
#   frozen). Replaces the earlier sequential 1a (60ep) + 1b (100ep). Joint training
#   converges faster since the heads co-adapt; 120 is a safe upper bound.
PHASE1_EPOCHS    = 120

PHASE2_EPOCHS    = 50    # classifier on PREDICTED p_f — converges fast; ES handles stopping

# PHASE3_EPOCHS=200: joint fine-tuning. With the lower P3 LRs, cosine decay
# over the full horizon produces steady gains; ES stops early if needed.
PHASE3_EPOCHS    = 200

# ─── Reproducibility ──────────────────────────────────────────────────────────
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    # H100 speed mode: drop strict determinism so cuDNN can autotune kernels.
    # Reproducibility within ~0.1% acc is preserved by manual seeds above.
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark     = True

print('Training configuration:')
print(f'  CKPT_PATH       : {CKPT_PATH}')
print(f'  HISTORY_PATH    : {HISTORY_PATH}')
print(f'  SEED={SEED}  |  Max epochs P1={PHASE1_EPOCHS} / P2={PHASE2_EPOCHS} / P3={PHASE3_EPOCHS}')
print(f'  Backbone lr: P1/P2={BACKBONE_LR}  P3={PHASE3_BACKBONE_LR}')
print(f'  Heads    lr: P1/P2={HEADS_LR}  P3={PHASE3_HEADS_LR}')
print(f'  WD: P1/P2={WEIGHT_DECAY}  P3={PHASE3_WD}')
print(f'  λ_coarse: P1/P3={LAMBDA_COARSE}/{PHASE3_LAMBDA_COARSE}  '
      f'λ_fine: P1/P3={LAMBDA_FINE}/{PHASE3_LAMBDA_FINE}  '
      f'λ_task: P1/P3={LAMBDA_TASK_P1}/{PHASE3_LAMBDA_TASK}')
print(f'  Label smoothing={LABEL_SMOOTHING}  Grad clip={GRAD_CLIP}  Warmup={WARMUP_EPOCHS} ep')
print(f'  LR plateau patience={LR_PATIENCE}  factor={LR_FACTOR}')
print(f'  ES patience: P1={PHASE1_PATIENCE} / P2&P3={EARLY_STOP_PAT}')
print(f'  Overfit threshold={OVERFIT_THRESHOLD} ({OVERFIT_PATIENCE} ep)')

Training configuration:
  CKPT_PATH       : /content/drive/MyDrive/XAI-Project/DB/DB1/checkpoints/best_model.pth
  HISTORY_PATH    : /content/drive/MyDrive/XAI-Project/DB/DB1/pipeline/train_history.pkl
  SEED=42  |  Max epochs P1=120 / P2=50 / P3=200
  Backbone lr: P1/P2=5e-05  P3=1e-05
  Heads    lr: P1/P2=0.0001  P3=3e-05
  WD: P1/P2=0.001  P3=0.005
  λ_coarse: P1/P3=1.0/2.0  λ_fine: P1/P3=1.0/2.0  λ_task: P1/P3=0.1/1.0
  Label smoothing=0.1  Grad clip=1.0  Warmup=5 ep
  LR plateau patience=7  factor=0.3
  ES patience: P1=20 / P2&P3=50
  Overfit threshold=0.2 (15 ep)


## 4. Loss Functions

`FocalLoss` and `compute_loss` are defined here (training-only utilities).
Per-part positive weights for the coarse head are computed from the training split loaded by `01_data.ipynb`.

```
L_total = λ_c · L_coarse + λ_f · L_fine + λ_t · L_task
```

| Loss | Formula | Masking |
|------|---------|---------|
| `L_coarse` | Weighted BCE on `z_c` vs L1 targets | part visibility |
| `L_fine`   | Focal Loss (α=0.25, γ=2) on `z_f` vs L2 targets | certainty ≥ 3 |
| `L_task`   | Cross-Entropy on `cls_logits` vs species label | none |


In [4]:
class FocalLoss(nn.Module):
    """Focal loss for handling class imbalance in binary attribute prediction."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets, mask=None):
        bce   = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt    = torch.exp(-bce)
        focal = self.alpha * (1 - pt) ** self.gamma * bce
        if mask is not None:
            focal = focal * mask
            return focal.sum() / mask.sum().clamp(min=1)
        return focal.mean()


def compute_loss(cls_out, z_c, p_c, z_f, p_f,
                 labels, l1, l2, cert, vis,
                 focal_fn, l1_pw, lam_c, lam_f, lam_t, label_smoothing=0.0):
    """Weighted sum of coarse, fine, and task losses."""
    # L_coarse: weighted BCE on z_c logits, masked by part visibility
    l_c = F.binary_cross_entropy_with_logits(z_c, l1, pos_weight=l1_pw, reduction='none')
    l_c = (l_c * vis).sum() / vis.sum().clamp(min=1)
    # L_fine: Focal loss on z_f logits, masked by certainty ≥ 3
    l_f = focal_fn(z_f, l2, mask=cert)
    # L_task: cross-entropy with optional label smoothing
    l_t = F.cross_entropy(cls_out, labels, label_smoothing=label_smoothing)
    return lam_c * l_c + lam_f * l_f + lam_t * l_t, l_c, l_f, l_t


# ── Per-part positive weights for L_coarse ────────────────────────────────────
l1_pos_weights_list = []
for j in range(NUM_L1):
    vis_j = train_visibility[:, j]
    l1_j  = train_l1[:, j]
    n_pos = float(((l1_j == 1) & (vis_j == 1)).sum())
    n_neg = float(((l1_j == 0) & (vis_j == 1)).sum())
    l1_pos_weights_list.append(n_neg / max(n_pos, 1.0))

l1_pos_weights = torch.tensor(l1_pos_weights_list, dtype=torch.float32, device=device)
focal_loss_fn  = FocalLoss(alpha=0.25, gamma=2.0)

print('L1 positive weights (N_neg/N_pos per part):')
for name, w in zip(CONCEPT_NAMES, l1_pos_weights.tolist()):
    print(f'  {name:12s}: {w:.2f}')


L1 positive weights (N_neg/N_pos per part):
  back        : 0.06
  belly       : 0.04
  bill        : 0.01
  breast      : 0.06
  crown       : 0.05
  eye         : 0.05
  forehead    : 0.05
  head        : 0.05
  leg         : 0.08
  nape        : 0.07
  tail        : 0.09
  throat      : 0.04
  wing        : 0.05


## 5. Training Utilities

`set_trainable`, `make_optimizer`, `make_scheduler`, `train_one_epoch`, `evaluate`,
`train_phase2_epoch`, and `eval_phase2` are defined here.


In [5]:

# ── H100 AMP: BF16 native (Hopper). No GradScaler is needed for BF16. ──────
_AMP_DTYPE = torch.bfloat16
torch.set_float32_matmul_precision('high')   # TF32 for fp32 matmuls on Hopper
# Enable Flash-Attention v2 / mem-efficient SDPA backends; disable math fallback
try:
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(False)
except Exception:
    pass
_bf16_ok = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False
print(f'AMP dtype : {_AMP_DTYPE}  (bf16_supported={_bf16_ok})  '
      f'matmul=high  flash_sdp=on')


def _orig(m):
    """Return the un-compiled module (after torch.compile wraps it)."""
    return m._orig_mod if hasattr(m, '_orig_mod') else m


def set_trainable(model, backbone, coarse, fine, classifier):
    """Freeze / unfreeze model components selectively (compile-safe)."""
    base = _orig(model)
    for p in base.features.parameters():    p.requires_grad = backbone
    for p in base.coarse_head.parameters(): p.requires_grad = coarse
    for p in base.fine_head.parameters():   p.requires_grad = fine
    for p in base.classifier.parameters():  p.requires_grad = classifier


def make_optimizer(model, backbone_lr, heads_lr, weight_decay):
    """Build two AdamW param-groups (compile-safe via id-based exclusion)."""
    base = _orig(model)
    backbone_params = [p for p in base.features.parameters() if p.requires_grad]
    bb_ids = {id(p) for p in backbone_params}
    head_params = [p for p in base.parameters()
                   if p.requires_grad and id(p) not in bb_ids]
    groups = []
    if backbone_params: groups.append({'params': backbone_params, 'lr': backbone_lr})
    if head_params:     groups.append({'params': head_params,     'lr': heads_lr})
    return torch.optim.AdamW(groups, weight_decay=weight_decay)


def make_scheduler(optimizer, patience, factor):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=patience, factor=factor
    )


class EarlyStopper:
    """
    Unified early stopping + overfitting detection for all three training phases.

    Stops training when EITHER condition is met:
      (a) val_loss has not improved by more than `min_delta` for `patience`
          consecutive epochs  → model has converged or is stuck.
      (b) train_acc − val_acc > `overfit_threshold` for `overfit_patience`
          consecutive epochs  → model is overfitting (only when threshold≠None).

    Usage:
        stopper = EarlyStopper(patience=20, overfit_threshold=0.15)
        # inside epoch loop:
        stopper.step(val_loss, train_acc, val_acc)
        if stopper.improved:
            save_checkpoint()
        if stopper.stop:
            break
    """
    def __init__(self, patience=20, min_delta=1e-4,
                 overfit_threshold=None, overfit_patience=10):
        self.patience          = patience
        self.min_delta         = min_delta
        self.overfit_threshold = overfit_threshold
        self.overfit_patience  = overfit_patience
        self.best              = float('inf')
        self.counter           = 0
        self.overfit_counter   = 0
        self.improved          = False
        self.stop              = False
        self.stop_reason       = ''

    def step(self, val_loss, train_acc=None, val_acc=None):
        # ── (a) val_loss improvement check ────────────────────────────────────
        if val_loss < self.best - self.min_delta:
            self.best     = val_loss
            self.counter  = 0
            self.improved = True
        else:
            self.counter += 1
            self.improved = False
            if self.counter >= self.patience:
                self.stop        = True
                self.stop_reason = f'no val_loss improvement for {self.patience} epochs'

        # ── (b) overfitting detection (Phase 3 only) ──────────────────────────
        if (self.overfit_threshold is not None
                and train_acc is not None and val_acc is not None):
            gap = train_acc - val_acc
            if gap > self.overfit_threshold:
                self.overfit_counter += 1
                if self.overfit_counter >= self.overfit_patience:
                    self.stop        = True
                    self.stop_reason = (
                        f'overfitting: train_acc−val_acc={gap:.3f} > '
                        f'{self.overfit_threshold} for {self.overfit_patience} epochs'
                    )
            else:
                self.overfit_counter = 0


def train_one_epoch(model, loader, optimizer, focal_fn, l1_pw, device,
                    lam_c, lam_f, lam_t, label_smoothing=0.0, grad_clip=None,
                    detach_coarse=False, scaler=None):
    """
    Train for one epoch.

    detach_coarse=True (Phase 1 only):
        Passes p_c.detach() to fine_head so that L_fine gradient cannot flow
        back through p_c into coarse_head.  Without this, the 312 fine-attribute
        gradients overwhelm the 13 coarse-attribute gradients and cause
        coarse_loss to diverge after a few epochs.
    """
    model.train()
    total_loss = total_c = total_f = total_t = 0.0
    correct = total = 0
    _use_amp = scaler is not None and device.type == 'cuda'
    for imgs, lbls, l1s, l2s, masks, certs, viss in loader:
        imgs  = imgs.to(device, non_blocking=True)
        lbls  = lbls.to(device, non_blocking=True)
        l1s   = l1s.to(device, non_blocking=True)
        l2s   = l2s.to(device, non_blocking=True)
        certs = certs.to(device, non_blocking=True)
        viss  = viss.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=_AMP_DTYPE, enabled=_use_amp):
            cls_out, z_c, p_c, z_f, p_f = model(imgs, detach_coarse=detach_coarse)
            loss, lc, lf, lt = compute_loss(cls_out, z_c, p_c, z_f, p_f,
                                             lbls, l1s, l2s, certs, viss,
                                             focal_fn, l1_pw, lam_c, lam_f, lam_t,
                                             label_smoothing=label_smoothing)
        if _use_amp:
            scaler.scale(loss).backward()
            if grad_clip is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], grad_clip
                )
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], grad_clip
                )
            optimizer.step()
        bs = lbls.size(0)
        total_loss += loss.item() * bs
        total_c    += lc.item()   * bs
        total_f    += lf.item()   * bs
        total_t    += lt.item()   * bs
        correct    += (cls_out.argmax(1) == lbls).sum().item()
        total      += bs
    return {'loss': total_loss/total, 'acc': correct/total,
            'coarse': total_c/total, 'fine': total_f/total, 'task': total_t/total}


def evaluate(model, loader, focal_fn, l1_pw, device, lam_c, lam_f, lam_t,
             label_smoothing=0.0):
    model.eval()
    total_loss = total_c = total_f = total_t = 0.0
    correct = total = 0
    _use_amp = device.type == 'cuda'
    with torch.no_grad():
        for imgs, lbls, l1s, l2s, masks, certs, viss in loader:
            imgs  = imgs.to(device, non_blocking=True)
            lbls  = lbls.to(device, non_blocking=True)
            l1s   = l1s.to(device, non_blocking=True)
            l2s   = l2s.to(device, non_blocking=True)
            certs = certs.to(device, non_blocking=True)
            viss  = viss.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=_AMP_DTYPE, enabled=_use_amp):
                cls_out, z_c, p_c, z_f, p_f = model(imgs)
                loss, lc, lf, lt = compute_loss(cls_out, z_c, p_c, z_f, p_f,
                                                 lbls, l1s, l2s, certs, viss,
                                                 focal_fn, l1_pw, lam_c, lam_f, lam_t,
                                                 label_smoothing=label_smoothing)
            bs = lbls.size(0)
            total_loss += loss.item() * bs
            total_c    += lc.item()   * bs
            total_f    += lf.item()   * bs
            total_t    += lt.item()   * bs
            correct    += (cls_out.argmax(1) == lbls).sum().item()
            total      += bs
    return {'loss': total_loss/total, 'acc': correct/total,
            'coarse': total_c/total, 'fine': total_f/total, 'task': total_t/total}


def train_phase2_epoch(model, loader, optimizer, device, label_smoothing=0.0, scaler=None):
    """
    Phase 2 (calibration): train classifier on the model's PREDICTED p_f
    (concept heads frozen).

    Rationale: in Phase 3 the classifier sees predicted p_f (continuous, attenuated
    by the hierarchical mask), not GT l2 ∈ {0,1}. Training on GT l2 here would
    create a distribution mismatch and collapse Phase 3 accuracy.
    """
    model.eval()                   # heads + backbone in eval (frozen, BN running stats)
    model.classifier.train()       # only classifier trainable
    total_loss = correct = total = 0
    _use_amp = scaler is not None and device.type == 'cuda'
    for imgs, lbls, l1s, l2s, masks, certs, viss in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=_AMP_DTYPE, enabled=_use_amp):
            with torch.no_grad():
                _, _, _, _, p_f = model(imgs)        # predicted concepts
            out  = model.classifier(p_f)             # classifier on predicted p_f
            loss = F.cross_entropy(out, lbls, label_smoothing=label_smoothing)
        if _use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()
        bs = lbls.size(0)
        total_loss += loss.item() * bs
        correct    += (out.argmax(1) == lbls).sum().item()
        total      += bs
    return {'loss': total_loss/total, 'acc': correct/total}


def eval_phase2(model, loader, device):
    """Phase 2: evaluate classifier with PREDICTED p_f (matches Phase 3 input)."""
    model.eval()
    total_loss = correct = total = 0
    _use_amp = device.type == 'cuda'
    with torch.no_grad():
        for imgs, lbls, l1s, l2s, masks, certs, viss in loader:
            imgs = imgs.to(device, non_blocking=True)
            lbls = lbls.to(device, non_blocking=True)
            with torch.autocast(device_type=device.type, dtype=_AMP_DTYPE, enabled=_use_amp):
                _, _, _, _, p_f = model(imgs)
                out  = model.classifier(p_f)
                loss = F.cross_entropy(out, lbls)
            bs = lbls.size(0)
            total_loss += loss.item() * bs
            correct    += (out.argmax(1) == lbls).sum().item()
            total      += bs
    return {'loss': total_loss/total, 'acc': correct/total}


# ── Verify losses on a dummy batch ────────────────────────────────────────────
model.eval()
with torch.no_grad():
    dummy_imgs   = torch.randn(4, 3, 224, 224, device=device)
    dummy_labels = torch.randint(0, 200,   (4,),           device=device)
    dummy_l1     = torch.randint(0, 2, (4, NUM_L1), dtype=torch.float32, device=device)
    dummy_l2     = torch.randint(0, 2, (4, NUM_L2), dtype=torch.float32, device=device)
    dummy_cert   = torch.randint(0, 2, (4, NUM_L2), dtype=torch.float32, device=device)
    dummy_vis    = torch.ones(4, NUM_L1, device=device)

    cls_out, z_c_out, p_c_out, z_f_out, p_f_out = model(dummy_imgs)
    l_total, lc, lf, lt = compute_loss(
        cls_out, z_c_out, p_c_out, z_f_out, p_f_out,
        dummy_labels, dummy_l1, dummy_l2, dummy_cert, dummy_vis,
        focal_loss_fn, l1_pos_weights,
        LAMBDA_COARSE, LAMBDA_FINE, LAMBDA_TASK,
        label_smoothing=LABEL_SMOOTHING
    )

print(f'Dummy batch loss verification:')
print(f'  L_coarse : {lc:.4f}')
print(f'  L_fine   : {lf:.4f}')
print(f'  L_task   : {lt:.4f}')
print(f'  L_total  : {l_total.item():.4f}')
print('All training utilities defined and verified.')
model.train()


AMP dtype : torch.bfloat16  (bf16_supported=True)  matmul=high  flash_sdp=on
Dummy batch loss verification:
  L_coarse : 0.3622
  L_fine   : 0.0437
  L_task   : 5.3300
  L_total  : 5.7359
All training utilities defined and verified.


HierarchicalCBM(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (0): Co

## 6. Run Training

Multi-phase training pipeline with **unified EarlyStopper** across all phases.

| Phase | Trainable components | Loss | Max Epochs | EarlyStopper |
|-------|----------------------|------|-----------|--------------|
| **1 — Joint heads** | `coarse_head` + `fine_head` + `classifier` (backbone frozen, `detach_coarse=False`) | λ_c·L_coarse + λ_f·L_fine + 0.1·L_task | ≤120 | val_loss plateau |
| **2 — Calibration** | `classifier` only (heads frozen, **predicted p_f** input) | L_task | ≤50 | val_loss plateau |
| **3 — Joint fine-tune** | all parameters (backbone unfrozen) | L_total | ≤200 | val_loss plateau **+ overfitting detection** |

**Phase 3 scheduler**: linear warmup (10 epochs) → cosine annealing to 1e-6.

**Key fixes vs the previous decoupled pipeline:**
1. Phase 2 trains the classifier on the model's **predicted** p_f (not GT `l2`), eliminating the train-test distribution mismatch that caused val_acc to collapse.
2. Phase 1 trains both concept heads **jointly** AND adds a small λ_task=0.1 so the concept space is shaped to be discriminative for classification from the start.
3. The hierarchical mask is **soft** (`0.5 + 0.5·p_c[parent]`), preventing `p_f → 0` early in training.
4. **336×336** resolution + **RandAugment** capture fine-grained bird parts that were lost at 224.

**Overfitting detection (Phase 3)**: if `train_acc − val_acc > 0.12` for 8 consecutive epochs → early stop.

In [6]:

# ══════════════════════════════════════════════════════════════════════════════
#  Multi-Phase Training Pipeline
#
#  Phase 1  — Joint head training  (backbone + cls FROZEN, both heads trainable)
#  Phase 2  — Classifier calibration (heads FROZEN, predicted p_f input)
#  Phase 3  — Joint fine-tuning   (all parameters unfrozen)
# ══════════════════════════════════════════════════════════════════════════════

import threading
from datetime import datetime

def _ts():
    return datetime.now().strftime('[%H:%M:%S]')

def _phase_banner(title):
    print()
    print('=' * 68)
    print(f'{_ts()}  {title}')
    print('=' * 68)

# ── GPU maximization ──────────────────────────────────────────────────────────
torch.backends.cudnn.benchmark    = True   # auto-tune kernels for fixed input size
torch.backends.cudnn.allow_tf32   = True   # TF32 on Ampere+ (A100, RTX 30xx+)
torch.backends.cudnn.deterministic = False  # disable determinism for speed

# H100 uses BF16 → no loss scaling needed. We pass a disabled GradScaler so the
# `_use_amp` branch in train_one_epoch still runs, but skips scale/unscale calls.
_scaler = torch.cuda.amp.GradScaler(enabled=False)
print(f'{_ts()}  AMP dtype={_AMP_DTYPE}  GradScaler enabled={_scaler.is_enabled()} (BF16, no scaling)')
print(f'{_ts()}  cudnn.benchmark=True  allow_tf32=True  matmul_precision=high  flash_sdp=on')

# torch.compile — Hopper benefits a lot from max-autotune (CUDA-graph + tuned
# kernels). First epoch JITs for ~3-5 min, then ~30-50% faster on H100.
if hasattr(torch, 'compile') and device.type == 'cuda':
    try:
        model = torch.compile(model, mode='max-autotune', fullgraph=False)
        print(f'{_ts()}  torch.compile(mode="max-autotune") applied — '
              'first epoch includes JIT compile')
    except Exception as _e:
        model = torch.compile(model)
        print(f'{_ts()}  torch.compile fallback (default mode): {_e}')
else:
    print(f'{_ts()}  torch.compile() skipped (PyTorch < 2.0 or CPU)')

# ══════════════════════════════════════════════════════════════════════════════
#  HEARTBEAT MONITOR
#  — Writes "still alive" line every HB_INTERVAL seconds to file only
#  — Last alive timestamp shown in each epoch summary line (not spammed to stdout)
#  — Check heartbeat file while disconnected:
#      Windows:  Get-Content <HB_FILE> -Tail 5
#      Linux  :  tail -5 <HB_FILE>
# ══════════════════════════════════════════════════════════════════════════════
_HB_INTERVAL = 60   # seconds
_HB_FILE     = os.path.join(CKPT_DIR, 'heartbeat.log')
_hb_state    = {'phase': 'init', 'epoch': 0, 'stop': False, 'last_alive': '—'}
_hb_lock     = threading.Lock()

def _hb_worker():
    while True:
        time.sleep(_HB_INTERVAL)
        with _hb_lock:
            if _hb_state['stop']:
                break
            ph = _hb_state['phase']
            ep = _hb_state['epoch']
            ts = _ts()
            _hb_state['last_alive'] = ts
        msg = f"{ts}  ♥  ALIVE — phase={ph}  epoch={ep}\n"
        # Write to file only — no stdout spam
        try:
            with open(_HB_FILE, 'a') as _hf:
                _hf.write(msg)
        except Exception:
            pass

def _hb_update(phase, epoch=0):
    with _hb_lock:
        _hb_state['phase'] = phase
        _hb_state['epoch'] = epoch

def _hb_stop():
    with _hb_lock:
        _hb_state['stop'] = True

# Start heartbeat thread
_hb_state['stop'] = False
_hb_thread = threading.Thread(target=_hb_worker, daemon=True, name='heartbeat')
_hb_thread.start()
with open(_HB_FILE, 'a') as _hf:
    _hf.write(f"{_ts()}  ── Training cell started ──\n")
print(f'{_ts()}  ♥ Heartbeat ON — every {_HB_INTERVAL}s  →  {_HB_FILE}')
print(f'{_ts()}    (last ♥ shown at end of each epoch summary line)')
print(f'{_ts()}    To check if alive while disconnected:')
print(f"{_ts()}      Get-Content '{_HB_FILE}' -Tail 5")

# ══════════════════════════════════════════════════════════════════════════════
#  PHASE HISTORY PRINTER
# ══════════════════════════════════════════════════════════════════════════════
def _show_phase_history(phase_key, col_specs, best_specs, n_tail=5):
    """
    Print best result + last n_tail epochs for a phase.

    col_specs  : list of (display_name, train_key, val_key)
    best_specs : list of (label, val_key, fn)  where fn = min or max
    """
    data = history[phase_key]
    n    = len(data.get(col_specs[0][1], []))
    if n == 0:
        print(f'{_ts()}    (no history yet for {phase_key})')
        return
    print(f'{_ts()}  ┌─ {phase_key} history — {n} epoch(s) done ──────────────────')
    # ── best results ──
    best_epochs = set()
    for label, val_key, fn in best_specs:
        vals = data[val_key]
        bv   = fn(vals)
        be   = vals.index(bv) + 1
        best_epochs.add(be)
        print(f'{_ts()}  │  Best {label:16s} = {bv:.4f}  @ epoch {be}')
    # ── skip earlier epochs, show last n_tail ──
    tail    = min(n_tail, n)
    skipped = n - tail
    if skipped > 0:
        print(f'{_ts()}  │  ... ({skipped} earlier epoch(s) not shown)')
    print(f'{_ts()}  │  Last {tail} epoch(s):')
    for i in range(n - tail, n):
        ep  = i + 1
        row = f'  ep {ep:3d}'
        for cname, tk, vk in col_specs:
            row += f'  {cname}: tr={data[tk][i]:.4f} vl={data[vk][i]:.4f}'
        if ep == n:
            row += '  ← LAST'
        if ep in best_epochs:
            row += '  ★BEST'
        print(f'{_ts()}  │ {row}')
    print(f'{_ts()}  └' + '─' * 56)

# ── Per-phase checkpoint paths ─────────────────────────────────────────────────
CKPT_1_PATH  = os.path.join(CKPT_DIR, 'ckpt_phase1_best.pth')
CKPT_2_PATH  = os.path.join(CKPT_DIR, 'ckpt_phase2_best.pth')
# CKPT_PATH & HISTORY_PATH defined in Cell 3

# ── Unified history dict (all phases) ─────────────────────────────────────────
_default_history = {
    'phase1':  {'train_loss': [], 'val_loss': [],
                'train_coarse': [], 'val_coarse': [],
                'train_fine':   [], 'val_fine':   []},
    'phase2':  {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []},
    'phase3':  {
        'train_loss': [], 'val_loss': [],
        'train_acc':  [], 'val_acc':  [],
        'train_coarse': [], 'val_coarse': [],
        'train_fine':   [], 'val_fine':   [],
        'train_task':   [], 'val_task':   [],
    },
}
if os.path.exists(HISTORY_PATH):
    with open(HISTORY_PATH, 'rb') as _fh:
        history = pickle.load(_fh)
    # Migrate from previous schema (phase1a/phase1b → fresh phase1)
    if 'phase1' not in history:
        history['phase1'] = _default_history['phase1']
        history.pop('phase1a', None)
        history.pop('phase1b', None)
        print(f'{_ts()}  Migrated history schema: phase1a/phase1b → phase1 (cleared)')
    # Ensure all expected keys are present
    for _ph, _d in _default_history.items():
        history.setdefault(_ph, _d)
    print(f'{_ts()}  Resumed history from {HISTORY_PATH}')
else:
    history = _default_history

def _save_history():
    with open(HISTORY_PATH, 'wb') as _fh:
        pickle.dump(history, _fh)


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 1 — Joint head training  (backbone + classifier FROZEN, both heads on)
# ══════════════════════════════════════════════════════════════════════════════
_phase_banner(f'Phase 1 — Joint Head Training  (≤{PHASE1_EPOCHS} ep | ES pat={PHASE1_PATIENCE})')
_show_phase_history(
    'phase1',
    col_specs  = [('coarse', 'train_coarse', 'val_coarse'),
                  ('fine',   'train_fine',   'val_fine')],
    best_specs = [('val_loss', 'val_loss', min)],
)

# Both concept heads + classifier trainable, detach_coarse=False so the heads
# co-adapt. Backbone remains frozen. A small λ_task=0.1 is added so the
# concept space is shaped to be discriminative for classification from the start.
set_trainable(model, backbone=False, coarse=True, fine=True, classifier=True)
opt_1   = make_optimizer(model, BACKBONE_LR, HEADS_LR, WEIGHT_DECAY)
sched_1 = make_scheduler(opt_1, LR_PATIENCE, LR_FACTOR)
stop_1  = EarlyStopper(patience=PHASE1_PATIENCE)

_hb_update('P1', 0)
print(f'{_ts()}  Phase 1 started — {PHASE1_EPOCHS} max epochs '
      f'(λ_c={LAMBDA_COARSE}  λ_f={LAMBDA_FINE}  λ_t={LAMBDA_TASK_P1})')
for ep in range(1, PHASE1_EPOCHS + 1):
    _hb_update('P1', ep)
    print(f'{_ts()}  P1 epoch {ep}/{PHASE1_EPOCHS} — training...', flush=True)
    t0  = time.time()
    trm = train_one_epoch(model, train_loader, opt_1,
                          focal_loss_fn, l1_pos_weights, device,
                          lam_c=LAMBDA_COARSE, lam_f=LAMBDA_FINE, lam_t=LAMBDA_TASK_P1,
                          label_smoothing=LABEL_SMOOTHING,
                          grad_clip=GRAD_CLIP, detach_coarse=False, scaler=_scaler)
    print(f'{_ts()}  P1 epoch {ep}/{PHASE1_EPOCHS} — evaluating...', flush=True)
    vlm = evaluate(model, val_loader, focal_loss_fn, l1_pos_weights, device,
                   lam_c=LAMBDA_COARSE, lam_f=LAMBDA_FINE, lam_t=LAMBDA_TASK_P1,
                   label_smoothing=LABEL_SMOOTHING)
    # Combined loss = L_coarse + L_fine + 0.1·L_task
    tr_loss = trm['coarse'] + trm['fine'] + LAMBDA_TASK_P1 * trm['task']
    vl_loss = vlm['coarse'] + vlm['fine'] + LAMBDA_TASK_P1 * vlm['task']
    sched_1.step(vl_loss)
    stop_1.step(vl_loss)
    history['phase1']['train_loss'].append(tr_loss)
    history['phase1']['val_loss'].append(vl_loss)
    history['phase1']['train_coarse'].append(trm['coarse'])
    history['phase1']['val_coarse'].append(vlm['coarse'])
    history['phase1']['train_fine'].append(trm['fine'])
    history['phase1']['val_fine'].append(vlm['fine'])

    marker = ' ✓' if stop_1.improved else f' ({stop_1.counter}/{PHASE1_PATIENCE})'
    print(f'{_ts()}  Best val_loss(c+f)={stop_1.best:.4f}')
    print(f'{_ts()}  P1 {ep:3d}/{PHASE1_EPOCHS} | '
          f'coarse tr={trm["coarse"]:.4f} vl={vlm["coarse"]:.4f} | '
          f'fine tr={trm["fine"]:.4f} vl={vlm["fine"]:.4f} | '
          f'lr={opt_1.param_groups[-1]["lr"]:.1e} | {time.time()-t0:.0f}s{marker} | '
          f'♥ {_hb_state["last_alive"]}')

    if stop_1.improved:
        torch.save({'epoch': ep, 'model_state_dict': model.state_dict(),
                    'val_coarse': vlm['coarse'], 'val_fine': vlm['fine']}, CKPT_1_PATH)
        print(f'{_ts()}  ✓ Checkpoint P1 saved — '
              f'val_coarse={vlm["coarse"]:.4f}  val_fine={vlm["fine"]:.4f}')
    if stop_1.stop:
        print(f'{_ts()}  [Early Stop P1] {stop_1.stop_reason}')
        break

_c1 = torch.load(CKPT_1_PATH, map_location=device, weights_only=False)
model.load_state_dict(_c1['model_state_dict'])
print(f'{_ts()}  → Loaded best P1  ep={_c1["epoch"]}  '
      f'val_coarse={_c1["val_coarse"]:.4f}  val_fine={_c1["val_fine"]:.4f}')
_save_history()


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2 — Classifier Calibration  (heads FROZEN, predicted p_f input)
# ══════════════════════════════════════════════════════════════════════════════
_phase_banner(f'Phase 2 — Sequential Classifier  (≤{PHASE2_EPOCHS} ep | ES pat={EARLY_STOP_PAT})')
_show_phase_history(
    'phase2',
    col_specs  = [('loss', 'train_loss', 'val_loss'), ('acc', 'train_acc', 'val_acc')],
    best_specs = [('val_loss', 'val_loss', min), ('val_acc', 'val_acc', max)],
)

set_trainable(model, backbone=False, coarse=False, fine=False, classifier=True)
opt_2   = make_optimizer(model, BACKBONE_LR, HEADS_LR, WEIGHT_DECAY)
sched_2 = make_scheduler(opt_2, LR_PATIENCE, LR_FACTOR)
stop_2  = EarlyStopper(patience=EARLY_STOP_PAT)

_hb_update('P2', 0)
print(f'{_ts()}  Phase 2 started — {PHASE2_EPOCHS} max epochs')
for ep in range(1, PHASE2_EPOCHS + 1):
    _hb_update('P2', ep)
    print(f'{_ts()}  P2 epoch {ep}/{PHASE2_EPOCHS} — training...', flush=True)
    t0  = time.time()
    trm = train_phase2_epoch(model, train_loader, opt_2, device,
                             label_smoothing=LABEL_SMOOTHING, scaler=_scaler)
    print(f'{_ts()}  P2 epoch {ep}/{PHASE2_EPOCHS} — evaluating...', flush=True)
    vlm = eval_phase2(model, val_loader, device)
    sched_2.step(vlm['loss'])
    stop_2.step(vlm['loss'])
    history['phase2']['train_loss'].append(trm['loss'])
    history['phase2']['val_loss'].append(vlm['loss'])
    history['phase2']['train_acc'].append(trm['acc'])
    history['phase2']['val_acc'].append(vlm['acc'])

    marker = ' ✓' if stop_2.improved else f' ({stop_2.counter}/{EARLY_STOP_PAT})'
    print(f'{_ts()}  Best val_loss={stop_2.best:.4f}')
    print(f'{_ts()}  P2 {ep:3d}/{PHASE2_EPOCHS} | '
          f'train  loss={trm["loss"]:.4f}  acc={trm["acc"]:.3f} | '
          f'val  loss={vlm["loss"]:.4f}  acc={vlm["acc"]:.3f} | '
          f'lr={opt_2.param_groups[-1]["lr"]:.1e} | {time.time()-t0:.0f}s{marker} | '
          f'♥ {_hb_state["last_alive"]}')

    if stop_2.improved:
        torch.save({'epoch': ep, 'model_state_dict': model.state_dict(),
                    'val_loss': vlm['loss'], 'val_acc': vlm['acc']}, CKPT_2_PATH)
        print(f'{_ts()}  ✓ Checkpoint P2 saved — val_loss={vlm["loss"]:.4f}')
    if stop_2.stop:
        print(f'{_ts()}  [Early Stop P2] {stop_2.stop_reason}')
        break

_c2 = torch.load(CKPT_2_PATH, map_location=device, weights_only=False)
model.load_state_dict(_c2['model_state_dict'])
print(f'{_ts()}  → Loaded best P2  ep={_c2["epoch"]}  val_loss={_c2["val_loss"]:.4f}  '
      f'val_acc={_c2["val_acc"]:.3f}')
_save_history()


# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 3 — Joint Fine-tuning  (all parameters unfrozen, full loss)
# ══════════════════════════════════════════════════════════════════════════════
_phase_banner(f'Phase 3 — Joint Fine-tuning  (≤{PHASE3_EPOCHS} ep | ES pat={EARLY_STOP_PAT} | OFT={OVERFIT_THRESHOLD})')
_show_phase_history(
    'phase3',
    col_specs  = [('loss', 'train_loss', 'val_loss'), ('acc', 'train_acc', 'val_acc')],
    best_specs = [('val_loss', 'val_loss', min), ('val_acc', 'val_acc', max)],
)

set_trainable(model, backbone=True, coarse=True, fine=True, classifier=True)
# Phase 3 uses dedicated lower LRs + stronger WD + concept-dominant loss weights
# so the model gently refines what P1+P2 already learned, instead of overfitting
# in the first few epochs (the symptom we saw with global LRs).
optimizer = make_optimizer(model, PHASE3_BACKBONE_LR, PHASE3_HEADS_LR, PHASE3_WD)

_warmup = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS
)
_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=max(PHASE3_EPOCHS - WARMUP_EPOCHS, 1), eta_min=1e-7
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[_warmup, _cosine], milestones=[WARMUP_EPOCHS]
)
stop_3 = EarlyStopper(
    patience=EARLY_STOP_PAT,
    overfit_threshold=OVERFIT_THRESHOLD,
    overfit_patience=OVERFIT_PATIENCE,
)

# ─── Best-val_acc tracking (independent of val_loss-based ES) ─────────────────
# IMPORTANT: With concept-dominant losses (λ_c=λ_f=2), val_loss is no longer
# monotonic with val_acc — coarse/fine BCE saturates at low values while task
# CE and val_acc keep evolving slowly. We therefore SAVE the checkpoint on
# best val_acc, but keep ES on val_loss so training continues as long as the
# model is still moving in concept-space.
best_val_acc_p3 = -1.0
best_acc_epoch  = 0

# ─── MixUp (Phase 3 only) ──────────────────────────────────────────────────────
# MixUp(α=0.2) is a strong regulariser for fine-grained classification: pairs of
# images and labels are linearly blended → smoother decision boundaries. Concept
# targets (l1, l2) and certainty/visibility masks are blended consistently.
# α=0.2 is conservative (typical CUB values are 0.1–0.4); higher α destroys
# concept-level structure that the heads need.
MIXUP_ALPHA = 0.2

def _mixup_batch(imgs, lbls, l1s, l2s, certs, viss, alpha=MIXUP_ALPHA):
    if alpha <= 0:
        return imgs, lbls, l1s, l2s, certs, viss, lbls, 1.0
    lam  = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(imgs.size(0), device=imgs.device)
    imgs = lam * imgs + (1 - lam) * imgs[perm]
    l1s  = lam * l1s  + (1 - lam) * l1s[perm]
    l2s  = lam * l2s  + (1 - lam) * l2s[perm]
    # For binary masks, take element-wise max so a concept is supervised whenever
    # EITHER source image had it visible/certain.
    certs = torch.maximum(certs, certs[perm])
    viss  = torch.maximum(viss,  viss[perm])
    return imgs, lbls, l1s, l2s, certs, viss, lbls[perm], lam


def train_p3_mixup_epoch(model, loader, optimizer, focal_fn, l1_pw, device,
                         lam_c, lam_f, lam_t, label_smoothing=0.0,
                         grad_clip=None, scaler=None, mixup_alpha=MIXUP_ALPHA):
    """Phase 3 training with MixUp on (image, l1, l2) and label-mixed CE."""
    model.train()
    total_loss = total_c = total_f = total_t = 0.0
    correct = total = 0
    _use_amp = scaler is not None and device.type == 'cuda'
    for imgs, lbls, l1s, l2s, masks, certs, viss in loader:
        imgs  = imgs.to(device, non_blocking=True)
        lbls  = lbls.to(device, non_blocking=True)
        l1s   = l1s.to(device, non_blocking=True)
        l2s   = l2s.to(device, non_blocking=True)
        certs = certs.to(device, non_blocking=True)
        viss  = viss.to(device, non_blocking=True)

        imgs_m, l_a, l1_m, l2_m, certs_m, viss_m, l_b, lam = _mixup_batch(
            imgs, lbls, l1s, l2s, certs, viss, alpha=mixup_alpha
        )

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=_AMP_DTYPE, enabled=_use_amp):
            cls_out, z_c, p_c, z_f, p_f = model(imgs_m, detach_coarse=False)
            # Concept losses on blended targets (continuous in [0,1])
            loss_blend, lc, lf, _ = compute_loss(
                cls_out, z_c, p_c, z_f, p_f,
                l_a, l1_m, l2_m, certs_m, viss_m,
                focal_fn, l1_pw, lam_c, lam_f, 0.0,   # task term computed below
                label_smoothing=0.0,
            )
            # MixUp CE for the task: linear combination of two cross-entropies
            ce_a = F.cross_entropy(cls_out, l_a, label_smoothing=label_smoothing)
            ce_b = F.cross_entropy(cls_out, l_b, label_smoothing=label_smoothing)
            l_t  = lam * ce_a + (1 - lam) * ce_b
            loss = loss_blend + lam_t * l_t

        if _use_amp:
            scaler.scale(loss).backward()
            if grad_clip is not None:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], grad_clip
                )
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward()
            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], grad_clip
                )
            optimizer.step()

        bs = lbls.size(0)
        total_loss += loss.item() * bs
        total_c    += lc.item()   * bs
        total_f    += lf.item()   * bs
        total_t    += l_t.item()  * bs
        # MixUp accuracy: weighted average of correct vs each source label
        preds       = cls_out.argmax(1)
        correct    += (lam * (preds == l_a).float() +
                       (1 - lam) * (preds == l_b).float()).sum().item()
        total      += bs
    return {'loss': total_loss/total, 'acc': correct/total,
            'coarse': total_c/total, 'fine': total_f/total, 'task': total_t/total}

_hb_update('P3', 0)
print(f'{_ts()}  Phase 3 started \u2014 {PHASE3_EPOCHS} max epochs '
      f'(bb_lr={PHASE3_BACKBONE_LR}  h_lr={PHASE3_HEADS_LR}  wd={PHASE3_WD}  '
      f'\u03bb=[{PHASE3_LAMBDA_COARSE},{PHASE3_LAMBDA_FINE},{PHASE3_LAMBDA_TASK}]  '
      f'mixup_\u03b1={MIXUP_ALPHA})')
for ep in range(1, PHASE3_EPOCHS + 1):
    _hb_update('P3', ep)
    print(f'{_ts()}  P3 epoch {ep}/{PHASE3_EPOCHS} \u2014 training (MixUp)...', flush=True)
    t0  = time.time()
    trm = train_p3_mixup_epoch(model, train_loader, optimizer,
                               focal_loss_fn, l1_pos_weights, device,
                               PHASE3_LAMBDA_COARSE, PHASE3_LAMBDA_FINE, PHASE3_LAMBDA_TASK,
                               label_smoothing=LABEL_SMOOTHING,
                               grad_clip=GRAD_CLIP, scaler=_scaler,
                               mixup_alpha=MIXUP_ALPHA)
    print(f'{_ts()}  P3 epoch {ep}/{PHASE3_EPOCHS} \u2014 evaluating...', flush=True)
    vlm = evaluate(model, val_loader, focal_loss_fn, l1_pos_weights, device,
                   PHASE3_LAMBDA_COARSE, PHASE3_LAMBDA_FINE, PHASE3_LAMBDA_TASK,
                   label_smoothing=LABEL_SMOOTHING)
    scheduler.step()
    stop_3.step(vlm['loss'], train_acc=trm['acc'], val_acc=vlm['acc'])

    history['phase3']['train_loss'].append(trm['loss'])
    history['phase3']['val_loss'].append(vlm['loss'])
    history['phase3']['train_acc'].append(trm['acc'])
    history['phase3']['val_acc'].append(vlm['acc'])
    history['phase3']['train_coarse'].append(trm['coarse'])
    history['phase3']['val_coarse'].append(vlm['coarse'])
    history['phase3']['train_fine'].append(trm['fine'])
    history['phase3']['val_fine'].append(vlm['fine'])
    history['phase3']['train_task'].append(trm['task'])
    history['phase3']['val_task'].append(vlm['task'])

    # ── Best val_acc tracking (independent of val_loss) ──────────────────────
    acc_improved = vlm['acc'] > best_val_acc_p3
    if acc_improved:
        best_val_acc_p3 = vlm['acc']
        best_acc_epoch  = ep

    lr_now  = optimizer.param_groups[-1]['lr']
    gap     = trm['acc'] - vlm['acc']
    gap_str = f'gap={gap:.3f}({stop_3.overfit_counter}/{OVERFIT_PATIENCE})'
    loss_marker = ' ✓L' if stop_3.improved else f' ({stop_3.counter}/{EARLY_STOP_PAT})'
    acc_marker  = ' ✓A' if acc_improved else ''
    print(f'{_ts()}  Best val_loss={stop_3.best:.4f}  '
          f'Best val_acc={best_val_acc_p3:.4f} @ ep{best_acc_epoch}')
    print(
        f'{_ts()}  P3 {ep:3d}/{PHASE3_EPOCHS} | '
        f'train  loss={trm["loss"]:.4f}  acc={trm["acc"]:.3f} '
        f'(c={trm["coarse"]:.3f} f={trm["fine"]:.3f} t={trm["task"]:.3f}) | '
        f'val  loss={vlm["loss"]:.4f}  acc={vlm["acc"]:.3f} | '
        f'{gap_str} | lr={lr_now:.2e} | {time.time()-t0:.0f}s{loss_marker}{acc_marker} | '
        f'♥ {_hb_state["last_alive"]}'
    )

    # Checkpoint on best val_acc (the metric users actually care about)
    if acc_improved:
        torch.save({
            'epoch':                ep,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss':             vlm['loss'],
            'val_acc':              vlm['acc'],
            'attr_parent_idx':      attr_parent_idx,
            'CONCEPT_NAMES':        CONCEPT_NAMES,
            'NUM_L1':               NUM_L1,
            'NUM_L2':               NUM_L2,
        }, CKPT_PATH)
        print(f'{_ts()}  ✓ Checkpoint saved — val_acc={vlm["acc"]:.4f}  '
              f'val_loss={vlm["loss"]:.4f}')
        _save_history()

    if stop_3.stop:
        print(f'{_ts()}  [Early Stop P3] {stop_3.stop_reason}')
        break

_hb_stop()
_save_history()
print(f'\n{_ts()}  Training complete.')
print(f'{_ts()}  Best Phase-3 val_acc  : {best_val_acc_p3:.4f} @ ep{best_acc_epoch}')
print(f'{_ts()}  Best Phase-3 val_loss : {stop_3.best:.4f}')
print(f'{_ts()}  Checkpoint            : {CKPT_PATH}')
print(f'{_ts()}  History               : {HISTORY_PATH}')
print(f'{_ts()}  Heartbeat log         : {_HB_FILE}')


[02:09:59]  AMP dtype=torch.bfloat16  GradScaler enabled=False (BF16, no scaling)
[02:09:59]  cudnn.benchmark=True  allow_tf32=True  matmul_precision=high  flash_sdp=on


/tmp/ipykernel_79980/579806011.py:28: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  _scaler = torch.cuda.amp.GradScaler(enabled=False)


[02:09:59]  torch.compile(mode="max-autotune") applied — first epoch includes JIT compile
[02:09:59]  ♥ Heartbeat ON — every 60s  →  /content/drive/MyDrive/XAI-Project/DB/DB1/checkpoints/heartbeat.log
[02:09:59]    (last ♥ shown at end of each epoch summary line)
[02:09:59]    To check if alive while disconnected:
[02:09:59]      Get-Content '/content/drive/MyDrive/XAI-Project/DB/DB1/checkpoints/heartbeat.log' -Tail 5

[02:09:59]  Phase 1 — Joint Head Training  (≤120 ep | ES pat=20)
[02:09:59]    (no history yet for phase1)
[02:09:59]  Phase 1 started — 120 max epochs (λ_c=1.0  λ_f=1.0  λ_t=0.1)
[02:09:59]  P1 epoch 1/120 — training...
[02:10:22]  P1 epoch 1/120 — evaluating...
[02:10:32]  Best val_loss(c+f)=0.6239
[02:10:32]  P1   1/120 | coarse tr=0.0725 vl=0.0719 | fine tr=0.0364 vl=0.0236 | lr=1.0e-04 | 32s ✓ | ♥ —
[02:10:32]  ✓ Checkpoint P1 saved — val_coarse=0.0719  val_fine=0.0236
[02:10:32]  P1 epoch 2/120 — training...
[02:10:37]  P1 epoch 2/120 — evaluating...
[02:10:38]  Be

## 7. Best Checkpoint Summary


In [7]:
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)

print('Best checkpoint summary:')
print(f'  Epoch         : {ckpt["epoch"]}')
print(f'  Val loss      : {ckpt["val_loss"]:.4f}')
print(f'  Val Top-1 acc : {ckpt["val_acc"]:.4f}  ({ckpt["val_acc"]*100:.1f}%)')

print(f'\nTo load in evaluate.ipynb:')
print(f'  ckpt = torch.load("{CKPT_PATH}", weights_only=False)')
print(f'  model.load_state_dict(ckpt["model_state_dict"])')


Best checkpoint summary:
  Epoch         : 127
  Val loss      : 2.5146
  Val Top-1 acc : 0.6417  (64.2%)

To load in evaluate.ipynb:
  ckpt = torch.load("/content/drive/MyDrive/XAI-Project/DB/DB1/checkpoints/best_model.pth", weights_only=False)
  model.load_state_dict(ckpt["model_state_dict"])
